In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import shutil

DRIVE_DIR  = "/content/drive/MyDrive/Processed_SP_ML"
LOCAL_DIR  = "/content/Processed_SP_ML"
RESULTS_DIR = "/content/drive/MyDrive/CNN_results"

# Copy dataset from Google Drive to Colab
if not os.path.exists(LOCAL_DIR):
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)

# Create results folder
os.makedirs(RESULTS_DIR, exist_ok=True)

# Count images
IMAGE_DIR = os.path.join(LOCAL_DIR, "images")
print("Images:", len(os.listdir(IMAGE_DIR)))

Images: 1032


In [3]:
import json, time, copy, random, warnings, glob
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, roc_curve
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NONE - turn on T4 GPU!")

device: cuda | GPU: Tesla T4


Sets ConvNeXt-Tiny as the pretrained model, defines image size, batch size, learning rates, epochs, and the five disease labels.

In [4]:
BACKBONE = "convnext_tiny"

PRETRAINED = True
IMG_W, IMG_H = 384, 288
EPOCHS, WARMUP_EPOCHS, PATIENCE = 30, 2, 7
BATCH_SIZE = 16                                  # if "CUDA out of memory" appears, change to 8
LR_HEAD, LR_BACKBONE, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
SEED = 42
LABEL_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]

RUN_DIR = os.path.join(RESULTS_DIR, f"image_only_{BACKBONE}")
os.makedirs(RUN_DIR, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

man = pd.read_csv(os.path.join(LOCAL_DIR, "Image_Manifest_MultiLabel.csv"))
train_df = man[man["Split"] == "train"].reset_index(drop=True)
N_FOLDS = int(train_df["CV_Fold"].max()) + 1
print(f"Model: {BACKBONE} | train images: {len(train_df)} | folds: {N_FOLDS}")
train_df.groupby("CV_Fold")[LABEL_COLS].sum()

Model: convnext_tiny | train images: 824 | folds: 5


,FECD,Guttata,Folds,PolyPleo,LowCD
CV_Fold,,,,,
0,20,16,61,22,20
1,21,17,62,21,21
2,21,16,61,24,23
3,20,16,61,20,20
4,19,16,61,18,18


Preprocess images and build the model
Resizes, augments, and normalizes grayscale images.

In [5]:
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
T = torchvision.transforms
TF_TRAIN = T.Compose([T.ToPILImage(),
                      T.RandomResizedCrop((IMG_H, IMG_W), scale=(0.7, 1.0), ratio=(1.25, 1.42)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
                      T.ColorJitter(brightness=0.2, contrast=0.2),
                      T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])
TF_EVAL = T.Compose([T.ToPILImage(), T.Resize((IMG_H, IMG_W)),
                     T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])

class SPDataset(Dataset):
    def __init__(self, frame, train):
        self.paths = [os.path.join(LOCAL_DIR, "images", f) for f in frame["Proc_File"]]
        self.Y = frame[LABEL_COLS].values.astype(np.float32)
        self.tf = TF_TRAIN if train else TF_EVAL
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE)), torch.tensor(self.Y[i])

class SimpleCNN(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], 3
        for c in channels:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers); self.pool = nn.AdaptiveAvgPool2d(1); self.out_dim = c_in
    def forward(self, x):
        return torch.flatten(self.pool(self.features(x)), 1)

def make_backbone(name, pretrained):
    W = "DEFAULT" if pretrained else None
    if name == "simple_cnn":
        m = SimpleCNN(); return m, m.out_dim, m.features
    if name == "densenet121":
        m = models.densenet121(weights=W); dim = m.classifier.in_features; m.classifier = nn.Identity(); cam = m.features
    elif name in ("efficientnet_b0", "efficientnet_b2"):
        m = getattr(models, name)(weights=W); dim = m.classifier[1].in_features; m.classifier = nn.Identity(); cam = m.features[-1]
    elif name == "resnet50":
        m = models.resnet50(weights=W); dim = m.fc.in_features; m.fc = nn.Identity(); cam = m.layer4
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=W); dim = m.classifier[2].in_features; m.classifier[2] = nn.Identity(); cam = m.features[-1]
    else:
        raise ValueError(name)
    return m, dim, cam

class MultiLabelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone, dim, self.cam_layer = make_backbone(BACKBONE, PRETRAINED)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim, len(LABEL_COLS)))
    def forward(self, x):
        return self.head(self.backbone(x))

n_params = sum(p.numel() for p in MultiLabelCNN().parameters())
print(f"OK - {BACKBONE} defined ({n_params/1e6:.2f} M parameters, pretrained={PRETRAINED})")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 185MB/s] 


OK - convnext_tiny defined (27.82 M parameters, pretrained=True)


Define training and prediction functions
Calculates positive-class weights to address disease-label imbalance, averages predictions from original and horizontally flipped images, and defines the training loop

In [6]:
def pos_weight(Y):
    pos = Y.sum(0); neg = len(Y) - pos
    return torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 10), dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for x, _ in loader:
        x = x.to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            p = torch.sigmoid(model(x)) + torch.sigmoid(model(torch.flip(x, dims=[3])))
        out.append((p / 2).float().cpu().numpy())
    return np.concatenate(out)

def macro_auc(Y, P):
    return np.mean([roc_auc_score(Y[:, j], P[:, j]) for j in range(Y.shape[1]) if 0 < Y[:, j].sum() < len(Y)])

def train_one(tr_frame, va_frame=None, epochs=EPOCHS, log=True):
    set_seed()
    tr_loader = DataLoader(SPDataset(tr_frame, True), batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, drop_last=True, pin_memory=True)
    va_loader = None if va_frame is None else DataLoader(SPDataset(va_frame, False), batch_size=32, num_workers=2)
    model = MultiLabelCNN().to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight(tr_frame[LABEL_COLS].values))
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": model.head.parameters(), "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - WARMUP_EPOCHS, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_auc, best_ep, wait, hist = None, -1, epochs, 0, []
    for ep in range(1, epochs + 1):
        warm = ep <= WARMUP_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad = not warm
        model.train(); t0, losses = time.time(), []
        for x, y in tr_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(loss.item())
        if not warm: sched.step()
        row = {"epoch": ep, "loss": np.mean(losses)}
        if va_loader is not None:
            P = predict(model, va_loader); row["val_macro_auc"] = macro_auc(va_frame[LABEL_COLS].values, P)
            if row["val_macro_auc"] > best_auc:
                best_auc, best_ep, best, wait = row["val_macro_auc"], ep, copy.deepcopy(model.state_dict()), 0
            elif not warm:
                wait += 1
        hist.append(row)
        if log:
            print(f"  ep {ep:2d} loss {row['loss']:.4f}" +
                  (f" | val macro-AUC {row['val_macro_auc']:.3f}" if va_loader else "") +
                  f"{' (warm-up)' if warm else ''} ({time.time() - t0:.0f}s)")
        if va_loader is not None and wait >= PATIENCE:
            break
    if best is not None: model.load_state_dict(best)
    return model, best_ep, pd.DataFrame(hist)

print("OK - training functions ready")

OK - training functions ready


In [7]:
oof_path  = os.path.join(RUN_DIR, "oof_predictions.csv")
info_path = os.path.join(RUN_DIR, "fold_info.json")
oof = pd.read_csv(oof_path) if os.path.exists(oof_path) else \
      train_df[["Tissue_ID", "CV_Fold"] + LABEL_COLS].assign(**{f"P_{l}": np.nan for l in LABEL_COLS})
fold_info = json.load(open(info_path)) if os.path.exists(info_path) else {}

for k in range(N_FOLDS):
    if str(k) in fold_info:
        print(f"Fold {k} already done (val macro-AUC {fold_info[str(k)]['val_macro_auc']:.3f}) - skipping"); continue
    tr_f = train_df[train_df["CV_Fold"] != k]; va_f = train_df[train_df["CV_Fold"] == k]
    print(f"\n===== {BACKBONE} | fold {k} | train {len(tr_f)} / val {len(va_f)} =====")
    t0 = time.time()
    model, best_ep, hist = train_one(tr_f, va_f)
    P = predict(model, DataLoader(SPDataset(va_f, False), batch_size=32, num_workers=2))
    oof.loc[oof["Tissue_ID"].isin(va_f["Tissue_ID"]), [f"P_{l}" for l in LABEL_COLS]] = P
    oof.to_csv(oof_path, index=False)
    hist.to_csv(os.path.join(RUN_DIR, f"history_fold{k}.csv"), index=False)
    fold_info[str(k)] = {"best_epoch": int(best_ep), "val_macro_auc": float(macro_auc(va_f[LABEL_COLS].values, P)),
                         "minutes": round((time.time() - t0) / 60, 1)}
    json.dump(fold_info, open(info_path, "w"), indent=2)
    print(f"Fold {k}: val macro-AUC {fold_info[str(k)]['val_macro_auc']:.3f}, best epoch {best_ep}, "
          f"{fold_info[str(k)]['minutes']} min - SAVED to Drive")
    del model; torch.cuda.empty_cache()


===== convnext_tiny | fold 0 | train 660 / val 164 =====
  ep  1 loss 1.1301 | val macro-AUC 0.702 (warm-up) (13s)
  ep  2 loss 1.0214 | val macro-AUC 0.701 (warm-up) (7s)
  ep  3 loss 1.1214 | val macro-AUC 0.701 (14s)
  ep  4 loss 0.9547 | val macro-AUC 0.827 (11s)
  ep  5 loss 0.8597 | val macro-AUC 0.818 (10s)
  ep  6 loss 0.8137 | val macro-AUC 0.825 (9s)
  ep  7 loss 0.8283 | val macro-AUC 0.801 (10s)
  ep  8 loss 0.8119 | val macro-AUC 0.810 (10s)
  ep  9 loss 0.7458 | val macro-AUC 0.797 (9s)
  ep 10 loss 0.6691 | val macro-AUC 0.784 (10s)
  ep 11 loss 0.6053 | val macro-AUC 0.832 (10s)
  ep 12 loss 0.5605 | val macro-AUC 0.790 (10s)
  ep 13 loss 0.7023 | val macro-AUC 0.800 (12s)
  ep 14 loss 0.5467 | val macro-AUC 0.787 (10s)
  ep 15 loss 0.6620 | val macro-AUC 0.812 (9s)
  ep 16 loss 0.4234 | val macro-AUC 0.799 (10s)
  ep 17 loss 0.4282 | val macro-AUC 0.775 (10s)
  ep 18 loss 0.3739 | val macro-AUC 0.803 (10s)
Fold 0: val macro-AUC 0.832, best epoch 11, 3.1 min - SAVED to

In [8]:
done = oof[[f"P_{l}" for l in LABEL_COLS]].notna().all(axis=1)
Y, P = oof.loc[done, LABEL_COLS].values, oof.loc[done, [f"P_{l}" for l in LABEL_COLS]].values
rows = {}
for j, lab in enumerate(LABEL_COLS):
    fpr, tpr, thr = roc_curve(Y[:, j], P[:, j]); b = np.argmax(tpr - fpr)
    yhat = (P[:, j] >= thr[b]).astype(int)
    rows[lab] = {"ROC_AUC": roc_auc_score(Y[:, j], P[:, j]), "PR_AUC": average_precision_score(Y[:, j], P[:, j]),
                 "F1": f1_score(Y[:, j], yhat), "Sensitivity": tpr[b], "Specificity": 1 - fpr[b]}
summary = pd.DataFrame(rows).T
summary.loc["Macro average"] = summary.mean()
print(f"{BACKBONE} - image only - train CV ({len(fold_info)}/{N_FOLDS} folds)")
display(summary.round(3))
summary.to_csv(os.path.join(RUN_DIR, "cv_metrics.csv"))
json.dump({"backbone": BACKBONE, "folds": fold_info, "macro_auc": float(summary.loc["Macro average", "ROC_AUC"])},
          open(os.path.join(RUN_DIR, "summary.json"), "w"), indent=2)

convnext_tiny - image only - train CV (5/5 folds)


,ROC_AUC,PR_AUC,F1,Sensitivity,Specificity
FECD,0.862,0.405,0.511,0.792,0.817
Guttata,0.870,0.519,0.472,0.790,0.830
Folds,0.828,0.708,0.705,0.784,0.739
PolyPleo,0.841,0.392,0.439,0.905,0.676
LowCD,0.863,0.423,0.456,0.922,0.701
Macro average,0.853,0.489,0.517,0.839,0.753


In [9]:
found = glob.glob("/content/drive/MyDrive/**/oof_predictions_numbers_only.csv", recursive=True)
NUM_OOF = found[0] if found else None
print("Numbers-only file:", NUM_OOF if NUM_OOF else "NOT FOUND")

table, ids = {}, set(train_df["Tissue_ID"])
if NUM_OOF:
    num = pd.read_csv(NUM_OOF); num = num[num["Tissue_ID"].isin(ids)]
    table["Numbers only (RF)"] = {l: roc_auc_score(num[l], num[f"P_{l}"]) for l in LABEL_COLS}
for d in sorted(os.listdir(RESULTS_DIR)):
    f = os.path.join(RESULTS_DIR, d, "oof_predictions.csv")
    if d.startswith("image_only_") and os.path.exists(f):
        o = pd.read_csv(f).dropna()
        if len(o) == len(train_df):
            table[d.replace("image_only_", "Image: ")] = {l: roc_auc_score(o[l], o[f"P_{l}"]) for l in LABEL_COLS}
comp = pd.DataFrame(table).T
comp["Macro AUC"] = comp.mean(axis=1)
comp.to_csv(os.path.join(RESULTS_DIR, "backbone_comparison.csv"))
print("Train CV ROC-AUC per disease (same folds for all)")
comp.round(3).sort_values("Macro AUC", ascending=False)

Numbers-only file: /content/drive/MyDrive/Processed_SP_ML/results_numbers_only/oof_predictions_numbers_only.csv
Train CV ROC-AUC per disease (same folds for all)


,FECD,Guttata,Folds,PolyPleo,LowCD,Macro AUC
Image: convnext_tiny,0.862,0.870,0.828,0.841,0.863,0.853
Image: densenet121,0.868,0.844,0.814,0.856,0.876,0.852
Image: efficientnet_b0,0.865,0.820,0.832,0.845,0.869,0.846
Image: resnet50,0.853,0.837,0.821,0.839,0.854,0.841
Image: simple_cnn,0.865,0.783,0.818,0.847,0.870,0.837
Numbers only (RF),0.983,0.554,0.633,0.963,0.971,0.821
